# Evidently AI — Monitoring Reports

Compares the reference population (original UCI dataset) against simulated production data from `notebooks/08_drift_simulation.ipynb` across three monitoring layers:

1. **Feature Drift** — which input distributions have shifted (`data_drift.html`)
2. **Data Quality** — missing values, range violations, stats (`data_quality.html`)
3. **Model Performance** — AUC, precision, recall against ground-truth labels (`model_performance.html`)

In [ ]:
import os
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('.').resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')


In [ ]:
from src.data_ingestion import load_data

# Reference: full UCI dataset (training population)
ref_df = load_data()
print(f"Reference data:  {ref_df.shape[0]:,} rows  |  "
      f"default rate: {ref_df['default'].mean():.1%}")

# Production: feature-drifted dataset from notebook 08
prod_path = PROJECT_ROOT / 'data' / 'simulated_production_data.csv'
if not prod_path.exists():
    raise FileNotFoundError(
        'Run notebooks/08_drift_simulation.ipynb first to generate '
        'data/simulated_production_data.csv'
    )
prod_df = pd.read_csv(prod_path)
print(f"Production data: {prod_df.shape[0]:,} rows  |  "
      f"default rate: {prod_df['default'].mean():.1%}")


In [ ]:
import joblib
from src.features.engineer import engineer_features

model = joblib.load(PROJECT_ROOT / 'models' / 'champion_model.joblib')
with open(PROJECT_ROOT / 'models' / 'champion_model_metadata.json') as f:
    metadata = json.load(f)

feature_names = metadata['feature_names']
threshold = metadata.get('threshold', 0.30)
TARGET_COL = 'default'


def _score(df):
    # engineer features then return champion model probability scores
    feats = engineer_features(df.drop(columns=[TARGET_COL], errors='ignore'))
    return pd.Series(model.predict_proba(feats[feature_names])[:, 1], index=df.index)


ref_df = ref_df.copy()
prod_df = prod_df.copy()
ref_df['prediction'] = _score(ref_df)
prod_df['prediction'] = _score(prod_df)

print(f'Reference  mean risk score: {ref_df["prediction"].mean():.4f}')
print(f'Production mean risk score: {prod_df["prediction"].mean():.4f}')
print(f'Classification threshold:   {threshold}')


In [ ]:
from evidently import Report, Dataset, BinaryClassification, DataDefinition
from evidently.presets import DataDriftPreset, DataSummaryPreset, ClassificationPreset

RAW_NUMERICAL = [
    'LIMIT_BAL', 'AGE',
    'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
    'PAY_AMT1',  'PAY_AMT2',  'PAY_AMT3',  'PAY_AMT4',  'PAY_AMT5',  'PAY_AMT6',
]
RAW_CATEGORICAL = [
    'SEX', 'EDUCATION', 'MARRIAGE',
    'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6',
]
FEATURE_COLS = RAW_NUMERICAL + RAW_CATEGORICAL
FULL_COLS = FEATURE_COLS + [TARGET_COL, 'prediction']

# Feature-only definition — for drift and quality reports
feature_def = DataDefinition(
    numerical_columns=RAW_NUMERICAL,
    categorical_columns=RAW_CATEGORICAL,
)

# Classification definition — for performance report
clf_def = DataDefinition(
    numerical_columns=RAW_NUMERICAL,
    categorical_columns=RAW_CATEGORICAL,
    classification=[
        BinaryClassification(
            target=TARGET_COL,
            prediction_probas='prediction',
            pos_label=1,
        )
    ],
)

ref_feat  = Dataset.from_pandas(ref_df[FEATURE_COLS],  data_definition=feature_def)
prod_feat = Dataset.from_pandas(prod_df[FEATURE_COLS], data_definition=feature_def)
ref_full  = Dataset.from_pandas(ref_df[FULL_COLS],     data_definition=clf_def)
prod_full = Dataset.from_pandas(prod_df[FULL_COLS],    data_definition=clf_def)

print('Datasets created.')
print(f'  Feature columns: {len(FEATURE_COLS)}')
print(f'  Numerical: {len(RAW_NUMERICAL)}  |  Categorical: {len(RAW_CATEGORICAL)}')


In [ ]:
REPORTS_DIR = PROJECT_ROOT / 'reports'
REPORTS_DIR.mkdir(exist_ok=True)

snapshot = Report(metrics=[DataDriftPreset()]).run(
    current_data=prod_feat,
    reference_data=ref_feat,
)
snapshot.save_html(str(REPORTS_DIR / 'data_drift.html'))
print('Saved: reports/data_drift.html')


In [ ]:
snapshot = Report(metrics=[DataSummaryPreset()]).run(
    current_data=prod_feat,
    reference_data=ref_feat,
)
snapshot.save_html(str(REPORTS_DIR / 'data_quality.html'))
print('Saved: reports/data_quality.html')


In [ ]:
snapshot = Report(metrics=[ClassificationPreset()]).run(
    current_data=prod_full,
    reference_data=ref_full,
)
snapshot.save_html(str(REPORTS_DIR / 'model_performance.html'))
print('Saved: reports/model_performance.html')


In [ ]:
from sklearn.metrics import roc_auc_score


def _psi(ref, cur, bins=10):
    # Population Stability Index
    lo = min(ref.min(), cur.min())
    hi = max(ref.max(), cur.max())
    breaks = np.linspace(lo, hi, bins + 1)
    ref_pct = np.histogram(ref, bins=breaks)[0] / len(ref) + 1e-8
    cur_pct = np.histogram(cur, bins=breaks)[0] / len(cur) + 1e-8
    return float(np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct)))


def _flag(value, alert, warn=None, higher_is_bad=True):
    if higher_is_bad:
        if value > alert:
            return '[ALERT]'
        if warn and value > warn:
            return '[WARN]'
        return '[OK]'
    return '[ALERT]' if value < alert else '[OK]'


# Feature drift
limit_psi = _psi(ref_df['LIMIT_BAL'], prod_df['LIMIT_BAL'])
age_psi   = _psi(ref_df['AGE'],       prod_df['AGE'])
_, ks_p   = stats.ks_2samp(ref_df['LIMIT_BAL'], prod_df['LIMIT_BAL'])

# Target drift
ref_rate  = ref_df[TARGET_COL].mean()
prod_rate = prod_df[TARGET_COL].mean()
_, target_ks_p = stats.ks_2samp(
    ref_df[TARGET_COL].astype(float),
    prod_df[TARGET_COL].astype(float),
)

# Concept drift proxy: correlation collapse of top SHAP driver
pay0_corr_ref  = ref_df['PAY_0'].corr(ref_df[TARGET_COL].astype(float))
pay0_corr_prod = prod_df['PAY_0'].corr(prod_df[TARGET_COL].astype(float))
corr_collapse  = abs(pay0_corr_ref - pay0_corr_prod)

# Model performance
ref_auc  = roc_auc_score(ref_df[TARGET_COL],  ref_df['prediction'])
prod_auc = roc_auc_score(prod_df[TARGET_COL], prod_df['prediction'])

SEP = '=' * 62
print(SEP)
print('MONITORING SUMMARY  --  Reference vs. Simulated Production')
print(SEP)

print('\nFEATURE DRIFT')
print(f'  LIMIT_BAL  PSI = {limit_psi:.3f}  {_flag(limit_psi, 0.25, 0.10)}')
print(f'  AGE        PSI = {age_psi:.3f}  {_flag(age_psi, 0.25, 0.10)}')
print(f"  LIMIT_BAL  KS p = {ks_p:.2e}  ({'significant' if ks_p < 0.05 else 'not significant'})")

print('\nTARGET DRIFT')
print(f'  Reference  default rate:  {ref_rate:.1%}')
print(f'  Production default rate:  {prod_rate:.1%}  ({prod_rate - ref_rate:+.1%} shift)')
print(f"  KS p = {target_ks_p:.2e}  ({'significant' if target_ks_p < 0.05 else 'not significant'})")

print('\nCONCEPT DRIFT  --  corr(PAY_0, default)')
print(f'  Reference:   {pay0_corr_ref:+.3f}')
print(f'  Production:  {pay0_corr_prod:+.3f}')
print(f'  Collapse:    {corr_collapse:.3f}  {_flag(corr_collapse, 0.15)}')

print('\nMODEL PERFORMANCE')
print(f'  Reference  AUC-ROC: {ref_auc:.4f}')
print(f'  Production AUC-ROC: {prod_auc:.4f}  {_flag(prod_auc, 0.72, higher_is_bad=False)}')
print(f'  AUC drop:           {ref_auc - prod_auc:+.4f}')

print('\nREPORTS GENERATED')
print('  reports/data_drift.html        -- per-feature drift visualization')
print('  reports/data_quality.html      -- missing values, range checks, stats')
print('  reports/model_performance.html -- AUC, precision, recall comparison')
